# Fine-Tuning Open Source LLMs

### Training Llama 3.2

Now we will fine tune the llama 3.2 with our datasets

If you are using LITE_MODE=True, then please run this on a free T4 box on colab.

If you are using LITE_MODE-False, then please use a paid A100 with high memory on colab.

In [ ]:
# Importing Libraries

import os
import re
import math
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

### Configure Project Settings

This cell defines the project configuration, including the base Llama model, dataset name, training mode, output model name, and all training hyperparameters. Keeping these values in one place makes the experiment easier to reproduce and modify.

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "Arivukkarasu" # your HF name here!

LITE_MODE = True

DATA_USER = "Arivukkarasu"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
  RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - overall

EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# Hyper-parameters - QLoRA

QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

# Hyper-parameters - training

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking

VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
LOG_TO_WANDB = True

In [ ]:
# A100 GPU supports this; T4 does not natively

use_bf16

# More on Optimizers

https://huggingface.co/docs/transformers/main/en/perf_train_gpu_one#optimizers

The most common is Adam or AdamW (Adam with Weight Decay).  
Adam achieves good convergence by storing the rolling average of the previous gradients; however, it adds an additional memory footprint of the order of the number of model parameters.


### Log in to HuggingFace and Weights & Biases

If you don't already have a HuggingFace account, visit https://huggingface.co to sign up and create a token.

Then select the Secrets for this Notebook by clicking on the key icon in the left, and add a new secret called `HF_TOKEN` with the value as your token.

Repeat this for weightsandbiases at https://wandb.ai and add a secret called `WANDB_API_KEY`

### Authenticate External Services

This cell logs into Hugging Face and Weights & Biases using securely stored API tokens. Authentication allows downloading pretrained models, uploading fine-tuned models, and tracking training metrics online.

In [ ]:
# Login to Hugging Face

hf_token = os.environ['HUGGING_FACE_WRITE_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
# Log in to Weights & Biases
wandb_api_key = os.environ['WANDB_API_KEY']
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

### Load Training, Validation, and Test Data

This cell loads the prepared dataset from the Hugging Face Hub and separates it into training, validation, and test splits. A smaller validation set is selected to speed up evaluation during training.

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE))
test = dataset['test']

In [ ]:
# if you wish to reduce the training dataset to 10,000 points instead, then uncomment this line:

# train = train.select(range(10000))

### Initialize Experiment Tracking

This cell starts a Weights & Biases run for logging training progress. Metrics such as loss, learning rate, and evaluation results will be recorded throughout the fine-tuning process.

In [ ]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

### Configure QLoRA Quantization

This cell sets up either 4-bit or 8-bit quantization using the BitsAndBytes library. Quantization significantly reduces GPU memory usage while maintaining good model performance during fine-tuning.

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

### Load the Pretrained Llama Model

This cell loads the tokenizer and pretrained Llama model from Hugging Face. It also applies the selected quantization settings and configures padding tokens required for text generation.

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

### Define LoRA Configuration

This cell specifies the Low-Rank Adaptation (LoRA) settings used during fine-tuning. LoRA updates only selected layers instead of the entire model, reducing memory requirements and training time.

In [ ]:
# LoRA Parameters

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

### Configure Training Parameters

This cell creates the training configuration, including batch size, learning rate, optimizer, checkpoint frequency, evaluation strategy, and model upload settings. These parameters control how the fine-tuning process is executed.

In [ ]:
# Training parameters

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    # if you are not using wandb try using tensorboard which is locally hosted, replace the below code for report_to
    # report_to=["tensorboard"],
    # logging_dir=f"./logs/{PROJECT_NAME}/{RUN_NAME}", # its the local directory
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS
)

### Initialize the Fine-Tuning Trainer

This cell creates an SFTTrainer object by combining the pretrained model, datasets, LoRA configuration, and training arguments. The trainer manages the complete supervised fine-tuning workflow.

In [ ]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)

### Fine-Tune the Model

This cell starts the supervised fine-tuning process using the configured trainer. During training, the model learns from the dataset while periodically evaluating performance and saving checkpoints.

In [ ]:
# Fine-tuning
fine_tuning.train()

### Save and Publish the Model

After training is complete, this cell uploads the fine-tuned model to the Hugging Face Hub. This makes the trained model available for future inference, sharing, or deployment.

In [ ]:
# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

### Close the Experiment

This cell ends the Weights & Biases logging session and ensures that all recorded metrics and artifacts are properly synchronized with the online dashboard.

In [ ]:
if LOG_TO_WANDB:
  wandb.finish()